# 2D/3D Structural Similarity Search

This notebook implements molecular similarity searches using RDKit.

Two comparison modes are available:
- **2D similarity:** compares molecular topology through fingerprints
  using the Tanimoto coefficient.
- **3D similarity:** compares molecular shapes after conformer alignment
  using shape Tanimoto similarity.

## Required inputs

- A reference molecule.
- A collection of candidate molecules.

Molecules may be supplied as:
- **2D topology**, such as a SMILES representation.
- **3D conformations**, such as molecular structures with 3D coordinates
  stored in an SDF file.

When only topology is available, conformations must be generated
before performing a 3D comparison.



## Loading molecular topology from SMILES

SMILES encodes atoms and their connectivity. It does not provide
3D coordinates.

The function below converts a user-supplied SMILES string into
an RDKit molecule. Invalid or empty inputs raise an explicit error.

In [1]:
from rdkit import Chem


def molecule_from_smiles(smiles):
    """Convert a SMILES string into an RDKit molecule."""
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("SMILES must be a non-empty string.")

    molecule = Chem.MolFromSmiles(smiles.strip())

    if molecule is None or molecule.GetNumAtoms() == 0:
        raise ValueError(f"Invalid molecular SMILES: {smiles!r}")

    return molecule

## Loading a supplied 3D conformation

An SDF file can store molecular connectivity and atomic coordinates,
and may contain multiple molecular records.

The function below loads a selected record and checks whether it
contains a conformer marked as 3D. Existing coordinates are preserved;
no conformers are generated or optimized here.

Record indices start at zero.

In [2]:
from pathlib import Path


def molecule_from_sdf(file_path, record_index=0):
    """Load a selected molecular record containing a 3D conformer."""
    path = Path(file_path)

    if not path.is_file():
        raise FileNotFoundError(f"SDF file not found: {path}")

    if not isinstance(record_index, int) or record_index < 0:
        raise ValueError("Record index must be a non-negative integer.")

    supplier = Chem.SDMolSupplier(str(path), removeHs=False)

    if record_index >= len(supplier):
        raise IndexError("Record index is outside the SDF file.")

    molecule = supplier[record_index]

    if molecule is None:
        raise ValueError("The selected molecular record could not be read.")

    if molecule.GetNumConformers() == 0:
        raise ValueError("The selected molecule has no coordinates.")

    if not molecule.GetConformer().Is3D():
        raise ValueError("The selected conformer is not marked as 3D.")

    return molecule

## Preparing a molecule for 3D comparison

This function returns a copy of the input molecule with 3D coordinates.

If conformers are already available, they must all be marked as 3D.
Their existing coordinates are preserved.
Existing conformers marked as 2D raise an explicit error.

If no conformers are available, a single conformer is generated
using ETKDGv3 after adding explicit hydrogens.


In [3]:
from rdkit.Chem import AllChem


def prepare_molecule_3d(molecule, random_seed=42):
    """Preserve supplied 3D coordinates or generate one conformer."""
    if not isinstance(molecule, Chem.Mol) or molecule.GetNumAtoms() == 0:
        raise ValueError("Input must be a non-empty RDKit molecule.")

    prepared_molecule = Chem.Mol(molecule)

    if prepared_molecule.GetNumConformers() > 0:
        if not all(
            conformer.Is3D()
            for conformer in prepared_molecule.GetConformers()
        ):
            raise ValueError(
                "Existing conformers must all be marked as 3D."
            )

        return prepared_molecule

    if (
        isinstance(random_seed, bool)
        or not isinstance(random_seed, int)
        or not 0 <= random_seed <= 2147483647
    ):
        raise ValueError("Random seed must be an integer from 0 to 2147483647.")

    prepared_molecule = Chem.AddHs(prepared_molecule)

    parameters = AllChem.ETKDGv3()
    parameters.randomSeed = random_seed
    parameters.numThreads = 1

    embedding_status = AllChem.EmbedMolecule(
        prepared_molecule,
        params=parameters
    )

    if embedding_status == -1:
        raise RuntimeError("3D conformer generation failed.")

    return prepared_molecule

In [4]:
example_molecule = molecule_from_smiles("CCO")
example_molecule_3d = prepare_molecule_3d(example_molecule)

print("Original conformers:", example_molecule.GetNumConformers())
print("Prepared conformers:", example_molecule_3d.GetNumConformers())
print("Is 3D:", example_molecule_3d.GetConformer().Is3D())

Original conformers: 0
Prepared conformers: 1
Is 3D: True


## Molecular alignment based on 3D shape

The probe molecule is aligned to the reference using RDKit's
rdShapeAlign module.

Alignment considers molecular shape only; chemical feature
("color") matching is disabled.

The selected conformers must contain 3D coordinates.
Input molecules are copied to preserve their original coordinates.

Shape Tanimoto similarity is calculated separately after alignment
using RDKit's grid-based shape representation.



In [5]:
from rdkit.Chem import rdShapeAlign


def align_molecules_by_shape(
    reference,
    probe,
    reference_conformer_id=-1,
    probe_conformer_id=-1
):
    """Align a probe to a reference using molecular shape only."""
    for molecule, conformer_id in (
        (reference, reference_conformer_id),
        (probe, probe_conformer_id)
    ):
        if not isinstance(molecule, Chem.Mol) or molecule.GetNumAtoms() == 0:
            raise ValueError("Inputs must be non-empty RDKit molecules.")

        if molecule.GetNumConformers() == 0:
            raise ValueError("Both molecules must have coordinates.")

        if not molecule.GetConformer(conformer_id).Is3D():
            raise ValueError("Selected conformers must be marked as 3D.")

    reference_copy = Chem.Mol(reference)
    aligned_probe = Chem.Mol(probe)

    rdShapeAlign.AlignMol(
        reference_copy,
        aligned_probe,
        refConfId=reference_conformer_id,
        probeConfId=probe_conformer_id,
        useColors=False
    )

    return reference_copy, aligned_probe


  

In [6]:
example_reference_3d = prepare_molecule_3d(
    molecule_from_smiles("CCO")
)

example_probe_3d = prepare_molecule_3d(
    molecule_from_smiles("CCCO")
)

aligned_reference, aligned_probe = align_molecules_by_shape(
    example_reference_3d,
    example_probe_3d
)

print("Shape-based alignment completed.")

Shape-based alignment completed.


## Calculating 3D shape Tanimoto similarity

Shape Tanimoto similarity measures the overlap of molecular shapes
represented on a spatial grid.

The input conformers must already be aligned. This function does not
perform alignment.

RDKit returns a shape Tanimoto distance, which is converted to similarity:

similarity = 1 - distance

Hydrogens are excluded from shape scoring. Grid spacing is configurable
and expressed in angstroms. Scores depend on the selected conformations,
their alignment, and the shape representation settings.

In [7]:
from rdkit.Chem import rdShapeHelpers


def shape_tanimoto_similarity(
    reference,
    aligned_probe,
    reference_conformer_id=-1,
    probe_conformer_id=-1,
    grid_spacing=0.5
):
    """Calculate shape similarity between already aligned conformers."""
    for molecule, conformer_id in (
        (reference, reference_conformer_id),
        (aligned_probe, probe_conformer_id)
    ):
        if not isinstance(molecule, Chem.Mol) or molecule.GetNumAtoms() == 0:
            raise ValueError("Inputs must be non-empty RDKit molecules.")

        if molecule.GetNumConformers() == 0:
            raise ValueError("Both molecules must have coordinates.")

        if not molecule.GetConformer(conformer_id).Is3D():
            raise ValueError("Selected conformers must be marked as 3D.")

    if grid_spacing <= 0:
        raise ValueError("Grid spacing must be positive.")

    distance = rdShapeHelpers.ShapeTanimotoDist(
        reference,
        aligned_probe,
        confId1=reference_conformer_id,
        confId2=probe_conformer_id,
        gridSpacing=grid_spacing,
        ignoreHs=True
    )

    return 1.0 - distance

In [8]:
example_similarity = shape_tanimoto_similarity(
    aligned_reference,
    aligned_probe
)

print("3D shape Tanimoto similarity:", example_similarity)

3D shape Tanimoto similarity: 0.7334437086092715


## Calculating 2D Tanimoto similarity

Molecular topology is encoded using binary Morgan fingerprints.
The fingerprints are compared using the Tanimoto coefficient.

Both molecules use the same fingerprint settings.
Radius, fingerprint size, and chirality encoding are configurable.
3D coordinates and molecular alignment are not required.

In [9]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator


def topology_tanimoto_similarity(
    reference,
    candidate,
    radius=2,
    fingerprint_size=2048,
    include_chirality=True
):
    """Compare molecular topology using binary Morgan fingerprints."""
    for molecule in (reference, candidate):
        if not isinstance(molecule, Chem.Mol) or molecule.GetNumAtoms() == 0:
            raise ValueError("Inputs must be non-empty RDKit molecules.")

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fingerprint_size,
        includeChirality=include_chirality
    )

    reference_fingerprint = generator.GetFingerprint(
        Chem.RemoveHs(reference)
    )
    candidate_fingerprint = generator.GetFingerprint(
        Chem.RemoveHs(candidate)
    )

    return DataStructs.TanimotoSimilarity(
        reference_fingerprint,
        candidate_fingerprint
    )

In [10]:
example_similarity_2d = topology_tanimoto_similarity(
    molecule_from_smiles("CCO"),
    molecule_from_smiles("CCCO")
)

print("2D Tanimoto similarity:", example_similarity_2d)

2D Tanimoto similarity: 0.5555555555555556


## Searching a collection by 2D similarity

Candidates are supplied as (compound identifier, RDKit molecule) pairs.
Identifiers should be unique so that each result can be traced to its input.

A single Morgan fingerprint generator is configured for the search.
The reference fingerprint is calculated once and compared with each
candidate fingerprint using the Tanimoto coefficient.

Results are sorted by decreasing similarity. Equal scores retain
the input order. Invalid molecules raise an explicit error; an empty
candidate collection returns an empty list.

In [11]:
def search_similarity_2d(
    reference,
    candidates,
    radius=2,
    fingerprint_size=2048,
    include_chirality=True
):
    """Rank candidate molecules by binary Morgan Tanimoto similarity."""
    if not isinstance(reference, Chem.Mol) or reference.GetNumAtoms() == 0:
        raise ValueError("Reference must be a non-empty RDKit molecule.")

    if (
        isinstance(radius, bool)
        or not isinstance(radius, int)
        or radius < 0
    ):
        raise ValueError("Radius must be a non-negative integer.")

    if (
        isinstance(fingerprint_size, bool)
        or not isinstance(fingerprint_size, int)
        or fingerprint_size <= 0
    ):
        raise ValueError("Fingerprint size must be a positive integer.")

    if not isinstance(include_chirality, bool):
        raise ValueError("Include chirality must be a boolean.")

    generator = rdFingerprintGenerator.GetMorganGenerator(
        radius=radius,
        fpSize=fingerprint_size,
        includeChirality=include_chirality
    )

    reference_fingerprint = generator.GetFingerprint(
        Chem.RemoveHs(reference)
    )

    results = []

    for compound_id, candidate in candidates:
        if not isinstance(candidate, Chem.Mol) or candidate.GetNumAtoms() == 0:
            raise ValueError(
                f"Candidate {compound_id!r} must be a non-empty RDKit molecule."
            )

        candidate_fingerprint = generator.GetFingerprint(
            Chem.RemoveHs(candidate)
        )

        similarity = DataStructs.TanimotoSimilarity(
            reference_fingerprint,
            candidate_fingerprint
        )

        results.append({
            "compound_id": compound_id,
            "similarity_2d": similarity
        })

    return sorted(
        results,
        key=lambda result: result["similarity_2d"],
        reverse=True
    )

In [12]:
example_reference = molecule_from_smiles("CCO")

example_candidates = [
    ("ethanol", molecule_from_smiles("CCO")),
    ("propanol", molecule_from_smiles("CCCO")),
    ("benzene", molecule_from_smiles("c1ccccc1"))
]

example_results_2d = search_similarity_2d(
    example_reference,
    example_candidates
)

for result in example_results_2d:
    print(result)

{'compound_id': 'ethanol', 'similarity_2d': 1.0}
{'compound_id': 'propanol', 'similarity_2d': 0.5555555555555556}
{'compound_id': 'benzene', 'similarity_2d': 0.0}


## Searching a collection by 3D shape similarity

Candidates are supplied as (compound identifier, RDKit molecule) pairs.

The reference is prepared once. Each candidate is prepared, aligned
to the reference using molecular shape only, and scored using
grid-based shape Tanimoto similarity.

Existing 3D coordinates are preserved during preparation.
If no conformers are available, one is generated using ETKDGv3.
Existing conformers must be marked as 3D.

Results are sorted by decreasing shape similarity.
Failures raise an explicit error identifying the candidate.

In [13]:
def search_similarity_3d(
    reference,
    candidates,
    random_seed=42,
    grid_spacing=0.5
):
    """Rank candidates by shape similarity using default conformers."""
    prepared_reference = prepare_molecule_3d(
        reference,
        random_seed=random_seed
    )

    results = []

    for compound_id, candidate in candidates:
        try:
            prepared_candidate = prepare_molecule_3d(
                candidate,
                random_seed=random_seed
            )

            aligned_reference, aligned_candidate = align_molecules_by_shape(
                prepared_reference,
                prepared_candidate
            )

            similarity = shape_tanimoto_similarity(
                aligned_reference,
                aligned_candidate,
                grid_spacing=grid_spacing
            )

        except (ValueError, RuntimeError) as error:
            raise RuntimeError(
                f"3D comparison failed for candidate {compound_id!r}: {error}"
            ) from error

        results.append({
            "compound_id": compound_id,
            "similarity_3d": similarity
        })

    return sorted(
        results,
        key=lambda result: result["similarity_3d"],
        reverse=True
    )

In [14]:
example_results_3d = search_similarity_3d(
    example_reference,
    example_candidates
)

for result in example_results_3d:
    print(result)

{'compound_id': 'ethanol', 'similarity_3d': 1.0}
{'compound_id': 'propanol', 'similarity_3d': 0.7334437086092715}
{'compound_id': 'benzene', 'similarity_3d': 0.5557644110275689}
